# 🛍️ E-Commerce Customer Churn Intelligence & MLOps Platform
## End-to-End Machine Learning, SHAP XAI, Survival Analysis, Causal Uplift, Algorithmic Fairness Audit & Monte Carlo Risk Simulation


In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Scikit-Learn 1.6+ compatibility patch for pickled XGBClassifier models
try:
    from sklearn.base import ClassifierMixin
    from sklearn.utils._tags import Tags, TargetTags, ClassifierTags
    ClassifierMixin.__sklearn_tags__ = lambda self: Tags(
        estimator_type='classifier',
        target_tags=TargetTags(required=False),
        transformer_tags=None,
        regressor_tags=None,
        classifier_tags=ClassifierTags()
    )
except Exception:
    pass

import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, recall_score, confusion_matrix
import joblib

print("All dependencies successfully imported!")


## 1. Dataset Loading & Exploratory Data Analysis (EDA)

In [ ]:
dataset_path = "mlops/data/ecommerce_customer_churn.csv"
if not os.path.exists(dataset_path):
    dataset_path = "mlops/data/bank_customer_churn.csv"

df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
print("\nTarget Class Distribution ('Churn'):")
print(df['Churn'].value_counts(normalize=True))
df.head()


## 2. Feature Preprocessing & Pipeline Construction

In [ ]:
target = 'Churn'

numeric_features = [
    'Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
    'SatisfactionScore', 'Complain', 'OrderAmountHikeFromlastYear',
    'DaySinceLastOrder', 'CashBackAmount', 'CityTier'
]

categorical_features = ['PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']

X = df[numeric_features + categorical_features].copy()
y = df[target].copy()

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Xtrain shape: {Xtrain.shape}, Xtest shape: {Xtest.shape}")

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
)

class_weight = float((len(ytrain) - sum(ytrain)) / max(1, sum(ytrain)))

xgb_clf = xgb.XGBClassifier(
    scale_pos_weight=class_weight,
    eval_metric='logloss',
    random_state=42
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_clf)
])

print("Pipeline constructed successfully!")


## 3. XGBoost Model Training & Hyperparameter Tuning

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1],
}

print("Starting Grid Search CV...")
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=1)
grid_search.fit(Xtrain, ytrain)

best_model = grid_search.best_estimator_
ypred = best_model.predict(Xtest)
ypred_prob = best_model.predict_proba(Xtest)[:, 1]

acc = accuracy_score(ytest, ypred)
rec = recall_score(ytest, ypred)
auc = roc_auc_score(ytest, ypred_prob)

print("\n--- Model Evaluation Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy : {acc:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nClassification Report:")
print(classification_report(ytest, ypred))


## 4. Explainable AI (SHAP & DiCE Counterfactual Recourse)

In [ ]:
from mlops.analytics.shap_explainer import calculate_shap_contributions
from mlops.analytics.counterfactual import generate_counterfactual_scenarios

sample_input = Xtest.iloc[[0]]
shap_df = calculate_shap_contributions(best_model, sample_input)

print("SHAP Feature Drivers for Sample Shopper:")
print(shap_df)

print("\n--- Counterfactual 'What-If' Recourse Scenarios ---")
scenarios = generate_counterfactual_scenarios(best_model, sample_input)
for sc in scenarios:
    print(f"\nScenario: {sc['Scenario_Name']}")
    print(f"Risk Reduction: {sc['Original_Risk_%']}% -> {sc['New_Risk_%']}% (Drop: {sc['Risk_Reduction_%']}%)")
    for act in sc['Actions_Required']:
        print(f"  - {act}")


## 5. Survival Analysis & 24-Month Retention Curves

In [ ]:
from mlops.analytics.survival_analysis import predict_survival_timeline

prob_val = float(best_model.predict_proba(sample_input)[0, 1])
surv_info = predict_survival_timeline(prob_val)

print("Survival Timeline Metrics:")
print(f"Expected Months Until Churn: {surv_info['Expected_Months_Until_Churn']} Months")
print(f"Hazard Risk Category       : {surv_info['Hazard_Risk_Category']}")
print(f"6-Month Survival Prob (%)  : {surv_info['Prob_Survival_6M_%']}%")
print(f"24-Month Survival Prob (%) : {surv_info['Prob_Survival_24M_%']}%")


## 6. Causal ML Uplift Modeling

In [ ]:
from mlops.analytics.uplift_modeling import segment_causal_uplift

test_preds = Xtest.copy()
test_preds['Churn_Probability'] = ypred_prob

uplift_df = segment_causal_uplift(test_preds)
print("Causal Uplift Segment Summary:")
print(uplift_df['Causal_Segment'].value_counts())


## 7. Algorithmic Fairness Audit (ECOA / 4/5th Rule)

In [ ]:
from mlops.analytics.fairness_audit import run_fairness_audit

fairness_res = run_fairness_audit(test_preds, protected_attribute='Gender')
print("Fairness Audit Summary:")
print(f"Protected Attribute     : {fairness_res['Protected_Attribute']}")
print(f"Disparate Impact Ratio  : {fairness_res['Disparate_Impact_Ratio']}")
print(f"Regulatory Status       : {fairness_res['Regulatory_Status']}")


## 8. Monte Carlo Revenue Risk Simulation

In [ ]:
from mlops.analytics.monte_carlo_sim import run_monte_carlo_simulation

mc_res = run_monte_carlo_simulation(test_preds, num_simulations=1000)
print("Monte Carlo Simulation Results (1,000 Trials):")
print(f"Mean Revenue Loss ($)      : ${mc_res['Mean_Revenue_Loss_USD']:,.2f}")
print(f"95% Value-at-Risk Loss ($) : ${mc_res['VaR_95_USD']:,.2f}")
print(f"Worst-Case Loss ($)        : ${mc_res['Worst_Case_Loss_USD']:,.2f}")


## 9. Model Serialization & Artifact Export

In [ ]:
joblib.dump(best_model, "best_churn_model.joblib")
print("Model pipeline successfully serialized to best_churn_model.joblib!")
